In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
from util import split_waveform_by_timestamps

token = os.environ.get("HUGGINGFACE_ACCESS_TOKEN")
audio_file = "sample.wav"

In [2]:
import whisperx

device = "cuda"
batch_size = 16  # reduce if low on GPU mem
compute_type = "float16"  # change to "int8" if low on GPU mem (may reduce accuracy)

# save model to local path (optional)
model_dir = "./data"
model = whisperx.load_model(
    "large-v2",
    device,
    compute_type=compute_type,
    download_root=model_dir,
    language="en",
)

/home/ansel/.pyenv/versions/3.12.3/lib/python3.12/site-packages/pyannote/audio/core/io.py:43: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.2.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/torch/whisperx-vad-segmentation.bin`


Model was trained with pyannote.audio 0.0.1, yours is 3.1.1. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.3.1+cu121. Bad things might happen unless you revert torch to 1.x.


In [4]:
audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)

In [5]:
# 2. Align whisper output
model_a, metadata = whisperx.load_align_model(
    language_code=result["language"], device=device
)
result = whisperx.align(
    result["segments"], model_a, metadata, audio, device, return_char_alignments=False
)

Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /home/ansel/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth
100%|██████████| 360M/360M [00:07<00:00, 51.3MB/s] 


In [6]:
# 3. Assign speaker labels
diarize_model = whisperx.DiarizationPipeline(use_auth_token=token, device=device)

# add min/max number of speakers if known
diarize_segments = diarize_model(audio, min_speakers=2, max_speakers=2)

result = whisperx.assign_word_speakers(diarize_segments, result)

In [7]:
timestamps = []
for segment in result["segments"]:
    if "speaker" in segment.keys():
        speaker = segment["speaker"]
    else:
        speaker = "unknown"
    timestamps.append((segment["start"], segment["end"], speaker))

In [8]:
from pyannote.audio import Audio

io = Audio(mono="downmix", sample_rate=16000)
waveform, sample_rate = io(audio_file)

In [9]:
split_waveform_by_timestamps(waveform, sample_rate, "whisper_out", timestamps)